# 02 · Feature Engineering

Show the two transform stages: row-level reformatting (`feature_engineer`) and rolling EWMA aggregation (`compute_rolling`). The key leakage-avoidance trick is the T+1 shift on rolling features — a row for GW N can only see stats from GWs 1..N-1.

In [1]:
from gaffer.providers.historical_csv import HistoricalCsvProvider
from gaffer.features.engineering import feature_engineer
from gaffer.features.rolling import compute_rolling

raw = HistoricalCsvProvider().get_historical_gwdata()
engineered = feature_engineer(raw)
engineered.head()

,season,was_home,team,opponent_team,team_fdr,opponent_team_fdr,team_points,team_goals_scored,team_goals_conceded,name,...,creativity,ict_index,influence,threat,value,selected_pct,transfers_balance_pct,total_points,bonus,bps
kickoff_date,,,,,,,,,,,,,,,,,,,,,
2019-08-31,2019-20,False,Brighton,Man City,2.0,5.0,0.0,0.0,4.0,Aaron Connolly,...,0.1,0.0,0.2,0.0,45,0.000000,0.000000,1,0,1
2019-09-14,2019-20,True,Brighton,Burnley,2.0,3.0,1.0,1.0,1.0,Aaron Connolly,...,0.3,2.2,1.0,21.0,45,0.001846,0.001393,1,0,1
2019-09-21,2019-20,False,Brighton,Newcastle,2.0,3.0,1.0,0.0,0.0,Aaron Connolly,...,4.8,2.5,2.0,18.0,45,0.003001,0.001064,1,0,1
2019-09-28,2019-20,False,Brighton,Chelsea,2.0,4.0,0.0,0.0,2.0,Aaron Connolly,...,0.6,0.1,0.2,0.0,45,0.004302,0.001110,1,0,2
2019-10-05,2019-20,True,Brighton,Spurs,2.0,4.0,3.0,3.0,0.0,Aaron Connolly,...,23.8,20.1,70.2,107.0,45,0.004609,0.000284,13,3,53


In [2]:
out = compute_rolling(engineered, alpha=0.3)
rolling = out.rolling
print('Rolling columns:', rolling.columns.tolist()[:8], '…')
rolling.head()

Rolling columns: ['minutes', 'yellow_cards', 'red_cards', 'assists', 'goals_scored', 'penalties_missed', 'goals_conceded', 'clean_sheets'] …


minutes  yellow_cards  red_cards   assists  \
name           kickoff_date                                                 
Aaron Connolly 2019-10-05          NaN           NaN        NaN       NaN   
               2019-10-19    41.152645           0.0        0.0  0.000000   
               2019-10-26    42.460749           0.0        0.0  0.000000   
               2019-11-02    54.406291           0.0        0.0  0.653847   
               2019-11-10    64.145871           0.0        0.0  0.445693   

                             goals_scored  penalties_missed  goals_conceded  \
name           kickoff_date                                                   
Aaron Connolly 2019-10-05             NaN               NaN             NaN   
               2019-10-19        0.721215               0.0        0.462695   
               2019-10-26        0.476001               0.0        0.645379   
               2019-11-02        0.320385               0.0        1.088237   
               2019-11-10        0.218390               0.0        0.741794   

                             clean_sheets  own_goals  saves  ...        bps  \
name           kickoff_date                                  ...              
Aaron Connolly 2019-10-05             NaN        NaN    NaN  ...        NaN   
               2019-10-19        0.360607        0.0    0.0  ...  20.004003   
               2019-10-26        0.238001        0.0    0.0  ...  14.222629   
               2019-11-02        0.160193        0.0    0.0  ...  16.765235   
               2019-11-10        0.427547        0.0    0.0  ...  12.064687   

                             ict_index  influence     threat      value  \
name           kickoff_date                                               
Aaron Connolly 2019-10-05          NaN        NaN        NaN        NaN   
               2019-10-19     7.987307  25.859515  44.362987  45.000000   
               2019-10-26     6.495619  18.563264  36.419554  45.000000   
               2019-11-02     4.437433  12.494495  25.166991  45.326924   
               2019-11-10     4.552857   8.516843  31.799229  45.541199   

                             selected_pct  transfers_balance_pct  team_points  \
name           kickoff_date                                                     
Aaron Connolly 2019-10-05             NaN                    NaN          NaN   
               2019-10-19        0.003506               0.000743     1.382208   
               2019-10-26        0.007278               0.003658     0.912256   
               2019-11-02        0.011189               0.003966     1.594789   
               2019-11-10        0.018654               0.007276     2.042141   

                             team_goals_scored  team_goals_conceded  
name           kickoff_date                                          
Aaron Connolly 2019-10-05                  NaN                  NaN  
               2019-10-19             1.205510             0.974866  
               2019-10-26             1.135636             1.323412  
               2019-11-02             1.745141             1.544605  
               2019-11-10             1.826276             1.052876  

[5 rows x 24 columns]

In [3]:
# Verify the T+1 shift: GW1 rolling row must be all-NaN (no prior history).
import pandas as pd

first_gws = rolling.groupby('name').head(1)
print('Fraction of all-NaN first rows:', first_gws.isna().all(axis=1).mean())

Fraction of all-NaN first rows: 1.0


**Next:** `03_model_benchmark.ipynb` — fit per-position models on these features.